# HD Integrated Visium Interactive Suite (HiVis) - demo notebook

This Notebook describes the first steps in the analysis of VisiumHD experiment using HiVis.

To see a details explanation and parameters for each function, visit the [documentation]().



In [ ]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from HiVis import HiVis

## Import data
The data in this tutorial is of mouse small intestine, available from [10X datasets](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-mouse-intestine), if you wish to follow along, download the following files:
* [Microscope image](https://cf.10xgenomics.com/samples/spatial-exp/3.0.0/Visium_HD_Mouse_Small_Intestine/Visium_HD_Mouse_Small_Intestine_tissue_image.btf) (1.6 GB)
* [Binned outputs](https://cf.10xgenomics.com/samples/spatial-exp/3.0.0/Visium_HD_Mouse_Small_Intestine/Visium_HD_Mouse_Small_Intestine_binned_outputs.tar.gz) (6 GB). Extract the .gz and .tar files. In this tutorial we will use the **square_002um** folder since it has the highest precision, but the **square_008um** and **square_016um** can be used the same way.

When creating a HiVis object, you need to specify the path of the data and image, and the name of the sample. Additionaly you can add any sample metadata as a dict, which is usefull when working with multiple samples. 

During the first import, which can take few minutes, the image is cropped to the actual coordinates of the data. The cropped image is the one that should be used later, when working with QuPath.

In [ ]:
path_image_fullres = r"outs\Visium_HD_Mouse_Small_Intestine_tissue_image.btf"
path_input_data = r"outs\binned_outputs\square_002um"
path_output = r"output"
properties = {"organism":"mouse",
              "organ":"Small intestine",
              "cancer": False,
              "source":"10X"}
si = HiVis.new(path_image_fullres, 
               path_input_data, 
               path_output,
               name="mouse_intestine",  
               properties=properties) # There are also filtering options, see documentation

We now have a HiVis object, which stores the images, anndata and enables plotting and basic analysis

In [ ]:
si 

## HiVis attributes

There are attributes we can access in HiVis.

In [ ]:
si.adata

In [ ]:
print(si.name)
print(si.image_fullres.shape) # Also image_highres, image_lowres
print(si.shape) # spots * genes
print(si.path_output)
print(si.properties)
print(si.json)

*****

Data can easily be acessed through the AnnData or directly

In [ ]:
print("nUMI: " + str(si["nUMI"])) # obs
print("nUMI_gene: " + str(si["nUMI_gene_log10"])) # var
gene = "Apob"
print("Apob: " + str(si[gene])) # gene data

## Subsetting the HiViz object
We can subset our object based on obs and/or var.

In this example we will subset only genes that are relatively highly expressed. 

The subsetting will create a new HiVis object.

In [ ]:
high_expressed = si["nUMI_gene"] > 10000
si_subset = si[:, high_expressed]
si_subset.rename("highly_expressed_genes") # otherwise it will be called "subset"

We can also subset based on OBS, for example by x,y coordinates.

In [ ]:
si_subset2 = si[(si["um_x" < 400) & (si["um_y" < 3600) & (si["um_y" > 3200),high_expressed]
si_subset2.qc()